# Chennai Reservoir Storage Analysis
**Assignment A0 — Domain Explorer | Part 5**  
**Student:** Bera Yashwant Pavan Kumar | **Cohort:** TechBharat C2-21

**Dataset:** Chennai Water Management by sudalairajkumar (Kaggle)  
**URL:** https://www.kaggle.com/datasets/sudalairajkumar/chennai-water-management  
**File:** `data/chennai_reservoir_levels.csv`  
**Columns:** `Date, POONDI, CHOLAVARAM, REDHILLS, CHEMBARAMBAKKAM` (units: **mcft** — million cubic feet)  
**Date format:** DD-MM-YYYY | **Date range:** 01-01-2004 to ~mid-2019 (daily)

---
**To run this notebook:**  
1. Download `chennai_reservoir_levels.csv` from the Kaggle link above.  
2. Place it at `water-intel/a0/bera_yashwant_pavan_kumar/data/chennai_reservoir_levels.csv`.  
3. Run all cells.


In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# -- Config -------------------------------------------------------------------
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         'data', 'chennai_reservoir_levels.csv')
RESERVOIRS = ['POONDI', 'CHOLAVARAM', 'REDHILLS', 'CHEMBARAMBAKKAM']
CAPACITIES = {
    'POONDI':          3231,
    'CHOLAVARAM':       881,
    'REDHILLS':        3300,
    'CHEMBARAMBAKKAM': 3645,
}  # mcft
TOTAL_CAP = sum(CAPACITIES.values())  # 11 057 mcft

COLORS = {
    'POONDI':          '#1f77b4',
    'CHOLAVARAM':      '#ff7f0e',
    'REDHILLS':        '#2ca02c',
    'CHEMBARAMBAKKAM': '#d62728',
}
DAY_ZERO = pd.Timestamp('2019-06-19')
print(f'Data path: {DATA_PATH}')
print(f'File exists: {os.path.exists(DATA_PATH)}')

In [ ]:
# -- Load & Preprocess --------------------------------------------------------
if not os.path.exists(DATA_PATH):
    print('\n[ACTION NEEDED]')
    print('Please download chennai_reservoir_levels.csv from:')
    print('  https://www.kaggle.com/datasets/sudalairajkumar/chennai-water-management')
    print(f'and place it at: {DATA_PATH}')
else:
    df = pd.read_csv(DATA_PATH, dayfirst=True, parse_dates=['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    df['TOTAL']     = df[RESERVOIRS].sum(axis=1)
    df['TOTAL_PCT'] = df['TOTAL'] / TOTAL_CAP * 100
    for res in RESERVOIRS:
        df[f'{res}_PCT'] = df[res] / CAPACITIES[res] * 100
    print(f'Loaded {len(df):,} rows | {df["Date"].min().date()} to {df["Date"].max().date()}')
    print(f'Columns: {list(df.columns[:6])}')
    df[RESERVOIRS].describe().round(1)

In [ ]:
# -- Plot: Individual Reservoir Levels + Combined % ---------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top panel: individual storage (mcft)
for res in RESERVOIRS:
    ax1.plot(df['Date'], df[res], label=res.title(), color=COLORS[res],
             linewidth=1.2, alpha=0.9)
ax1.set_ylabel('Storage (mcft)', fontsize=11)
ax1.set_title(
    'Chennai Four-Reservoir Daily Storage (2004 -- mid-2019)\n'
    'Source: CMWSSB via Kaggle (sudalairajkumar/chennai-water-management)',
    fontsize=12
)
ax1.axvline(DAY_ZERO, color='black', linestyle=':', linewidth=1.5)
ax1.text(DAY_ZERO, ax1.get_ylim()[1] * 0.90, ' Day Zero\n 19 Jun 2019',
         fontsize=8, color='black',
         bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', ec='grey', alpha=0.8))
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Bottom panel: combined % of total capacity
ax2.plot(df['Date'], df['TOTAL_PCT'], color='navy', linewidth=1.5,
         label='Combined level (% of 11,057 mcft total)')
ax2.fill_between(df['Date'], df['TOTAL_PCT'], alpha=0.15, color='navy')
ax2.axvspan(pd.Timestamp('2018-01-01'), pd.Timestamp('2019-06-30'),
            color='red', alpha=0.08, label='2018--2019 drought period')
ax2.axvline(DAY_ZERO, color='black', linestyle=':', linewidth=1.5)
ax2.text(DAY_ZERO, 50, ' Day Zero\n 19 Jun 2019',
         fontsize=8, color='black',
         bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', ec='grey', alpha=0.8))
recovery = pd.Timestamp('2019-11-01')
ax2.axvline(recovery, color='green', linestyle='--', linewidth=1.3)
ax2.text(recovery, 5, ' NE Monsoon\n 2019 recovery',
         fontsize=8, color='darkgreen',
         bbox=dict(boxstyle='round,pad=0.2', fc='honeydew', ec='green', alpha=0.8))
ax2.axhline(10, color='red', linestyle='--', linewidth=1, alpha=0.7, label='10% emergency threshold')
ax2.set_ylabel('Combined Level (% of capacity)', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylim(0, 110)
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
out = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'chennai_reservoirs.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out}')

## 5-Line Interpretation

**(a) Annual NE monsoon pattern:** Reservoir levels reliably recover each October–December during the northeast monsoon, filling from seasonal lows (~20–40 %) to peaks of 70–100 % — the city's primary recharge window — before declining through the dry summer months.

**(b) 2018–2019 divergence:** The 2018 northeast monsoon received only ~344 mm against a long-period average of ~758 mm (≈55 % deficit), following sub-par monsoons in 2016 and 2017; combined storage crashed from ~40 % in Oct 2018 to effectively 0 % by June 2019 — a trajectory with no historical precedent in the 2004–2019 record.

**(c) Early-warning opportunity:** A monitoring system tracking the October post-monsoon fill rate could have flagged by **November 2018** that the seasonal recharge was critically low; issuing a Tier-1 alert by December 2018 (storage < 25 % of capacity with no recovery trend) would have given authorities a **6-month window** to implement demand rationing, activate emergency tanker/groundwater contingency plans, and fast-track inter-basin transfers before Day Zero in June 2019.